In [ ]:
import os
import time
from datetime import datetime

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("WARSAW_API_KEY")

URL = "https://api.um.warszawa.pl/api/action/busestrams_get/"
RESOURCE_ID = "f2e5503e-927d-4ad3-9500-4ab9e55deb59"

print("Key loaded:", API_KEY is not None)

Key loaded: True


In [ ]:
def fetch_vehicles(vehicle_type):
    """vehicle_type: 1 = buses, 2 = trams"""
    params = {
        "resource_id": RESOURCE_ID,
        "apikey": API_KEY,
        "type": vehicle_type,
    }
    response = requests.get(URL, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    result = data["result"]
    if not isinstance(result, list):
        raise ValueError(f"API returned an error: {result}")
    return result

trams = fetch_vehicles(2)
print(len(trams))
trams[:2]

308


[{'Lines': '28',
  'Lon': 20.931065,
  'VehicleNumber': '1290',
  'Time': '2026-09-26 00:32:27',
  'Lat': 52.298904,
  'Brigade': '6'},
 {'Lines': '9',
  'Lon': 21.0554,
  'VehicleNumber': '1294',
  'Time': '2026-09-26 00:50:22',
  'Lat': 52.256203,
  'Brigade': '012'}]

In [4]:
all_records = []

for i in range(30):
    poll_time = datetime.now()
    try:
        records = fetch_vehicles(2)
        for r in records:
            r["poll_time"] = poll_time
        all_records.extend(records)
        print(f"Poll {i + 1}/30: {len(records)} records")
    except Exception as e:
        print(f"Poll {i + 1}/30 failed: {e}")
    time.sleep(10)

df = pd.DataFrame(all_records)
df.shape

Poll 1/30: 309 records
Poll 2/30: 308 records
Poll 3/30: 308 records
Poll 4/30: 309 records
Poll 5/30: 308 records
Poll 6/30: 308 records
Poll 7/30: 309 records
Poll 8/30: 309 records
Poll 9/30: 308 records
Poll 10/30: 307 records
Poll 11/30: 309 records
Poll 12/30: 308 records
Poll 13/30: 308 records
Poll 14/30: 309 records
Poll 15/30: 306 records
Poll 16/30: 308 records
Poll 17/30: 307 records
Poll 18/30: 305 records
Poll 19/30: 306 records
Poll 20/30: 308 records
Poll 21/30: 306 records
Poll 22/30: 307 records
Poll 23/30: 307 records
Poll 24/30: 307 records
Poll 25/30: 308 records
Poll 26/30: 307 records
Poll 27/30: 307 records
Poll 28/30: 306 records
Poll 29/30: 307 records
Poll 30/30: 306 records


(9225, 7)

In [7]:
df.head()
df.dtypes

Lines                    object
Lon                     float64
VehicleNumber            object
Time                     object
Lat                     float64
Brigade                  object
poll_time        datetime64[ns]
dtype: object

In [11]:
updates = df.groupby("VehicleNumber")["Time"].nunique()
updates.value_counts().sort_index()

Time
1     118
6       1
8       1
10      1
16      1
23      1
27      2
28      3
29      3
30    180
Name: count, dtype: int64

In [13]:
df["Time"] = pd.to_datetime(df["Time"])
df["staleness"] = df["poll_time"] - df["Time"]
df["staleness"].describe()

df.sort_values("staleness", ascending=False).head(10)

,Lines,Lon,VehicleNumber,Time,Lat,Brigade,poll_time,staleness
8925,17,20.928076,1316,2024-10-26 07:09:54,52.299427,016,2026-09-26 20:51:44.757998,700 days 13:41:50.757998
8618,17,20.928076,1316,2024-10-26 07:09:54,52.299427,016,2026-09-26 20:51:34.521657,700 days 13:41:40.521657
8312,17,20.928076,1316,2024-10-26 07:09:54,52.299427,016,2026-09-26 20:51:24.308180,700 days 13:41:30.308180
8005,17,20.928076,1316,2024-10-26 07:09:54,52.299427,016,2026-09-26 20:51:14.085065,700 days 13:41:20.085065
7698,17,20.928076,1316,2024-10-26 07:09:54,52.299427,016,2026-09-26 20:51:03.843295,700 days 13:41:09.843295
7390,17,20.928076,1316,2024-10-26 07:09:54,52.299427,016,2026-09-26 20:50:53.619481,700 days 13:40:59.619481
7083,17,20.928076,1316,2024-10-26 07:09:54,52.299427,016,2026-09-26 20:50:43.383053,700 days 13:40:49.383053
6776,17,20.928076,1316,2024-10-26 07:09:54,52.299427,016,2026-09-26 20:50:33.173662,700 days 13:40:39.173662
6469,17,20.928076,1316,2024-10-26 07:09:54,52.299427,016,2026-09-26 20:50:22.969100,700 days 13:40:28.969100
6163,17,20.928076,1316,2024-10-26 07:09:54,52.299427,016,2026-09-26 20:50:12.749423,700 days 13:40:18.749423


In [14]:
last_poll = df[df["poll_time"] == df["poll_time"].max()].copy()
print(len(last_poll))  # should be around 307, one row per tram

last_poll.sort_values("staleness", ascending=False).head(15)

306


,Lines,Lon,VehicleNumber,Time,Lat,Brigade,poll_time,staleness
8925,17,20.928076,1316,2024-10-26 07:09:54,52.299427,016,2026-09-26 20:51:44.757998,700 days 13:41:50.757998
9202,16,20.999866,4255,2025-02-09 23:33:12,52.186535,18,2026-09-26 20:51:44.757998,593 days 21:18:32.757998
9210,14,21.000126,4264,2025-02-28 22:29:26,52.186520,2,2026-09-26 20:51:44.757998,574 days 22:22:18.757998
8969,7,21.055262,3011,2025-04-29 11:50:32,52.256115,2,2026-09-26 20:51:44.757998,515 days 09:01:12.757998
8948,24,20.972492,2018,2026-09-08 10:28:26,52.232620,016,2026-09-26 20:51:44.757998,18 days 10:23:18.757998
8963,19,20.988680,2102,2026-09-24 10:20:08,52.247677,018,2026-09-26 20:51:44.757998,2 days 10:31:36.757998
8954,20,20.971910,2044,2026-09-25 09:57:03,52.231804,013,2026-09-26 20:51:44.757998,1 days 10:54:41.757998
8938,7,21.053713,1428,2026-09-25 18:20:01,52.255688,010,2026-09-26 20:51:44.757998,1 days 02:31:43.757998
9059,2,20.931430,3240,2026-09-25 18:20:11,52.299633,024,2026-09-26 20:51:44.757998,1 days 02:31:33.757998
9070,2,20.931145,3254,2026-09-25 18:34:33,52.299320,025,2026-09-26 20:51:44.757998,1 days 02:17:11.757998


In [15]:
last_poll["staleness"].describe()

count                           306
mean      8 days 02:34:16.646886888
std      68 days 05:34:20.003095858
min          0 days 00:00:13.757998
25%          0 days 00:00:17.757998
50%          0 days 00:00:20.757998
75%          0 days 20:09:53.507998
max        700 days 13:41:50.757998
Name: staleness, dtype: object

In [16]:
thresholds = {
    "2 min": pd.Timedelta(minutes=2),
    "10 min": pd.Timedelta(minutes=10),
    "1 hour": pd.Timedelta(hours=1),
    "1 day": pd.Timedelta(days=1),
}

for label, limit in thresholds.items():
    count = (last_poll["staleness"] > limit).sum()
    print(f"Older than {label}: {count} trams")

Older than 2 min: 118 trams
Older than 10 min: 112 trams
Older than 1 hour: 95 trams
Older than 1 day: 37 trams


In [17]:
updates = df.groupby("VehicleNumber")["Time"].nunique()
frozen_ids = updates[updates == 1].index

last_poll["is_frozen"] = last_poll["VehicleNumber"].isin(frozen_ids)

last_poll.groupby("is_frozen")["staleness"].describe()

,count,mean,std,min,25%,50%,75%,max
is_frozen,,,,,,,,
False,190,0 days 00:00:20.794840105,0 days 00:00:23.512997880,0 days 00:00:13.757998,0 days 00:00:16.757998,0 days 00:00:17.757998,0 days 00:00:19.757998,0 days 00:04:14.757998
True,116,21 days 09:15:22.266618690,109 days 19:45:11.964751004,0 days 00:05:09.757998,0 days 01:28:19.007998,0 days 21:08:20.757998,1 days 00:51:41.257998,700 days 13:41:50.757998


In [18]:
lat_bad = (last_poll["Lat"] < 51.9) | (last_poll["Lat"] > 52.5)
lon_bad = (last_poll["Lon"] < 20.6) | (last_poll["Lon"] > 21.5)

out_of_bounds = last_poll[lat_bad | lon_bad]
print("Out of bounds:", len(out_of_bounds))
out_of_bounds

Out of bounds: 0


,Lines,Lon,VehicleNumber,Time,Lat,Brigade,poll_time,staleness,is_frozen


In [19]:
print("Missing:", last_poll[["Lat", "Lon"]].isna().sum().to_dict())
print("Zeros:", ((last_poll["Lat"] == 0) | (last_poll["Lon"] == 0)).sum())

Missing: {'Lat': 0, 'Lon': 0}
Zeros: 0


In [20]:
last_poll[["Lat", "Lon"]].describe()

,Lat,Lon
count,306.000000,306.000000
mean,52.246564,20.994225
std,0.039766,0.047789
min,52.157257,20.900955
25%,52.225251,20.957784
50%,52.244706,20.995590
75%,52.273754,21.024438
max,52.341430,21.119278


In [21]:
# All trams in the last poll
print("Duplicate VehicleNumber:", last_poll["VehicleNumber"].duplicated().sum())
print("Duplicate Lines + Brigade:", last_poll[["Lines", "Brigade"]].duplicated().sum())

Duplicate VehicleNumber: 0
Duplicate Lines + Brigade: 24


In [22]:
# Only live trams
live = last_poll[~last_poll["is_frozen"]]
print("Live, duplicate VehicleNumber:", live["VehicleNumber"].duplicated().sum())
print("Live, duplicate Lines + Brigade:", live[["Lines", "Brigade"]].duplicated().sum())

Live, duplicate VehicleNumber: 0
Live, duplicate Lines + Brigade: 0


In [23]:
dupes = last_poll[last_poll[["Lines", "Brigade"]].duplicated(keep=False)]
dupes.sort_values(["Lines", "Brigade"])

,Lines,Lon,VehicleNumber,Time,Lat,Brigade,poll_time,staleness,is_frozen
9099,1,21.000807,3607,2026-09-26 00:32:06,52.187588,1,2026-09-26 20:51:44.757998,0 days 20:19:38.757998,True
9103,1,21.000498,3612,2026-09-26 20:51:30,52.186264,1,2026-09-26 20:51:44.757998,0 days 00:00:14.757998,False
8932,10,20.971783,1392,2026-09-25 21:52:17,52.231860,9,2026-09-26 20:51:44.757998,0 days 22:59:27.757998,True
9057,10,21.021784,3237,2026-09-26 20:51:25,52.180645,9,2026-09-26 20:51:44.757998,0 days 00:00:19.757998,False
9171,11,20.998854,4213,2026-09-26 20:51:29,52.232346,3,2026-09-26 20:51:44.757998,0 days 00:00:15.757998,False
9179,11,21.001106,4225,2026-09-25 20:05:14,52.187714,3,2026-09-26 20:51:44.757998,1 days 00:46:30.757998,True
9200,14,21.021774,4253,2026-09-26 20:51:29,52.210970,2,2026-09-26 20:51:44.757998,0 days 00:00:15.757998,False
9210,14,21.000126,4264,2025-02-28 22:29:26,52.186520,2,2026-09-26 20:51:44.757998,574 days 22:22:18.757998,True
8923,15,20.931803,1310,2026-09-25 21:02:25,52.299545,4,2026-09-26 20:51:44.757998,0 days 23:49:19.757998,True
9150,15,20.980500,4019,2026-09-26 20:51:29,52.275192,4,2026-09-26 20:51:44.757998,0 days 00:00:15.757998,False
